# 의미 검색과 검색어 일치를 함께 비교하기

임베딩 검색은 표현이 달라도 비슷한 의미를 찾습니다. 검색어 일치는 부품 이름처럼 정확한 표현을 찾는 데 도움이 됩니다. 이번에는 두 점수를 같은 비중으로 합칩니다. 이처럼 여러 검색 방법을 함께 쓰는 것을 하이브리드 검색이라고 합니다.

기존 질문 20개는 그대로 유지하고, 다른 표현의 질문 10개도 따로 비교합니다. 질문 결과를 보고 비중을 반복해서 맞추지는 않습니다. 이번 결과는 작은 자료 묶음에 대한 실험입니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 수업의 의미 검색과 TF-IDF 개념을 결합한 개인 검색 실험입니다.
- 추가: 글자 2~4개 단위 TF-IDF와 의미 점수를 50%씩 결합합니다. 점수는 정답 확률이 아닙니다.

전체 목록: [수업 기반·추가 적용 기록](../src/zzong_santafe_lag/learning_additions.md)


### 1. 기존 검색 실험 불러오기

06번의 코드 셀을 실행해 기존 벡터와 평가 질문을 준비합니다. 자료와 벡터는 메모리에만 만들어집니다. 06번 파일의 코드가 바뀌면 이 노트북 결과도 달라질 수 있습니다. Python 파일로 정리할 때 이 연결을 함수로 바꿉니다.

In [1]:
# [프로젝트 추가 기록] 수업의 의미 검색과 TF-IDF 개념을 결합한 개인 검색 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import contextlib
import io

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
source_path = project_folder / "notebooks" / "06_embedding_parent_retrieval.ipynb"
source_notebook = json.loads(source_path.read_text(encoding="utf-8"))
previous = {}
with contextlib.redirect_stdout(io.StringIO()):
    for index, cell in enumerate(source_notebook["cells"]):
        if cell["cell_type"] == "code":
            exec(compile("".join(cell["source"]), f"{source_path.name}:cell_{index}", "exec"), previous)

np = previous["np"]
parent_lookup = previous["parent_lookup"]
search_chunks = previous["search_chunks"]
chunk_vectors = previous["chunk_vectors"]
embedding_model = previous["embedding_model"]
token_count = previous["token_count"]
token_budget = previous["token_budget"]
evaluation_questions = previous["evaluation_questions"]
question_vectors = previous["question_vectors"]
rank_parents = previous["rank_parents"]
parent_ids = list(parent_lookup)
print("원문 묶음:", len(parent_ids))
print("기존 질문:", len(evaluation_questions))
print("기존 의미 검색 첫 결과 일치:", sum(r["top1_correct"] for r in previous["evaluation_results"]))

원문 묶음: 5
기존 질문: 20
기존 의미 검색 첫 결과 일치: 15


### 2. 검색어 일치용 색인 만들기

색인은 검색할 표현과 자료를 연결하는 목록입니다. 한글 띄어쓰기가 달라도 비교할 수 있도록 공백과 문장부호를 제외하고, 두~네 글자씩 이어진 조각을 셉니다. 예를 들어 '패들 쉬프트'와 '패들쉬프트'를 같은 형태로 비교할 수 있습니다.

TF-IDF는 자주 나오는 표현보다 특정 자료에 있는 표현에 더 큰 무게를 주는 계산입니다. 안내도에서는 부품 이름을 색인에 넣고, 참조 쪽수와 공통 안내 문구는 제외합니다. 원문 자체는 그대로 유지합니다.

In [2]:
# [프로젝트 추가 기록] 수업의 의미 검색과 TF-IDF 개념을 결합한 개인 검색 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
import re
import math
from collections import Counter

def normalize_keyword(text):
    """한글·영문·숫자를 남겨 띄어쓰기와 문장부호의 차이를 줄입니다."""
    return "".join(re.findall(r"[가-힣A-Za-z0-9]+", text)).lower()

def char_terms(text):
    """정리한 글에서 연속된 2~4글자 조각을 셉니다."""
    text = normalize_keyword(text)
    return Counter(text[i:i+n] for n in [2, 3, 4] for i in range(max(0, len(text)-n+1)))

# 안내도는 실제 부품 이름을 검색합니다. 원문과 그림은 parent_lookup에 그대로 있습니다.
lexical_texts = []
for parent_id in parent_ids:
    parent = parent_lookup[parent_id]
    items = parent["metadata"].get("navigation_items")
    if items:
        lexical_texts.append("\n".join(item["name"] for item in items))
    else:
        lexical_texts.append(parent["content"])

document_terms = [char_terms(text) for text in lexical_texts]
document_frequency = Counter(term for counts in document_terms for term in counts)
vocabulary = {term: index for index, term in enumerate(sorted(document_frequency))}
idf = np.array([math.log((1 + len(parent_ids)) / (1 + document_frequency[term])) + 1
                for term in vocabulary], dtype=np.float32)

def lexical_vector(text):
    """글자 조각 빈도와 IDF로 벡터를 만들고 길이를 1로 맞춥니다."""
    vector = np.zeros(len(vocabulary), dtype=np.float32)
    for term, count in char_terms(text).items():
        if term in vocabulary:
            index = vocabulary[term]
            vector[index] = (1 + math.log(count)) * idf[index]
    length = np.linalg.norm(vector)
    return vector / length if length else vector

lexical_matrix = np.stack([lexical_vector(text) for text in lexical_texts])
print("검색어 색인 크기:", lexical_matrix.shape)
print("자료에 없는 표현에는 별도 동의어나 정답을 추가하지 않았습니다.")

검색어 색인 크기: (5, 2719)
자료에 없는 표현에는 별도 동의어나 정답을 추가하지 않았습니다.


### 3. 두 검색 점수 합치기

의미 검색과 검색어 일치 점수를 각각 50%씩 반영합니다. 이 비중은 실험 시작 전에 정한 값이며 최적값이라는 뜻은 아닙니다. 전체 PDF에서 별도 질문으로 다시 확인해야 합니다.

부품 이름 전체가 질문에 그대로 포함된 경우도 기록해 결과를 살펴볼 수 있게 합니다. 답변은 아직 생성하지 않습니다.

In [3]:
# [프로젝트 추가 기록] 수업의 의미 검색과 TF-IDF 개념을 결합한 개인 검색 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
semantic_weight = 0.5

def hybrid_rank(question, query_vector, top_k=3):
    """의미 검색과 검색어 일치를 합쳐 원문 묶음 후보를 돌려줍니다."""
    keyword_vector = lexical_vector(question)
    keyword_scores = lexical_matrix @ keyword_vector
    chunk_scores = chunk_vectors @ query_vector
    results = []

    for parent_index, parent_id in enumerate(parent_ids):
        indices = [i for i, chunk in enumerate(search_chunks)
                   if chunk["metadata"]["parent_record_id"] == parent_id]
        best_index = max(indices, key=lambda i: chunk_scores[i])
        semantic_score = float(chunk_scores[best_index])
        keyword_score = float(keyword_scores[parent_index])
        parent = parent_lookup[parent_id]
        exact_names = [
            item["name"] for item in parent["metadata"].get("navigation_items", [])
            if normalize_keyword(item["name"]) in normalize_keyword(question)
        ]
        results.append({
            "parent_record": parent,
            "matched_chunk": search_chunks[best_index],
            "semantic_score": semantic_score,
            "keyword_score": keyword_score,
            "combined_score": semantic_weight * semantic_score + (1 - semantic_weight) * keyword_score,
            "exact_names": exact_names,
        })
    results.sort(key=lambda item: item["combined_score"], reverse=True)
    return results[:top_k]

def hybrid_search(question, top_k=3):
    """질문 길이를 확인하고 하이브리드 검색 후보를 가져옵니다."""
    if not question.strip():
        raise ValueError("질문을 입력하세요.")
    if token_count(question) > token_budget:
        raise ValueError("질문이 너무 깁니다. 짧게 나눠 주세요.")
    if not 1 <= top_k <= len(parent_ids):
        raise ValueError("검색 결과 수를 원문 묶음 수 안에서 지정하세요.")
    vector = embedding_model.encode(
        question, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
    )
    return hybrid_rank(question, vector, top_k=top_k)

def compare_questions(questions, vectors):
    """같은 질문에서 두 방법의 순위를 비교하고, 개선과 악화를 함께 기록합니다."""
    rows = []
    for (question, expected_id), vector in zip(questions, vectors):
        semantic_hits = rank_parents(vector, top_k=3)
        hybrid_hits = hybrid_rank(question, vector, top_k=3)
        semantic_ids = [hit["parent_record"]["metadata"]["record_id"] for hit in semantic_hits]
        hybrid_ids = [hit["parent_record"]["metadata"]["record_id"] for hit in hybrid_hits]
        rows.append({
            "question": question, "expected_id": expected_id,
            "semantic_first": semantic_ids[0], "hybrid_first": hybrid_ids[0],
            "semantic_top1": semantic_ids[0] == expected_id,
            "hybrid_top1": hybrid_ids[0] == expected_id,
            "hybrid_top3": expected_id in hybrid_ids,
        })
    return rows

def print_comparison(rows, label):
    """비교 결과와 바뀐 순위를 출력합니다."""
    print(label, "| 질문 수:", len(rows))
    print("의미 검색 첫 결과 일치:", sum(r["semantic_top1"] for r in rows))
    print("결합 검색 첫 결과 일치:", sum(r["hybrid_top1"] for r in rows))
    print("결합 검색 상위 세 결과 포함:", sum(r["hybrid_top3"] for r in rows))
    for row in rows:
        if row["semantic_first"] != row["hybrid_first"] or not row["hybrid_top1"]:
            print("질문:", row["question"])
            print("기대:", row["expected_id"], "| 의미:", row["semantic_first"], "| 결합:", row["hybrid_first"])

original_comparison = compare_questions(evaluation_questions, question_vectors)
print_comparison(original_comparison, "기존 질문")

기존 질문 | 질문 수: 20
의미 검색 첫 결과 일치: 15
결합 검색 첫 결과 일치: 20
결합 검색 상위 세 결과 포함: 20
질문: 유리창 잠금 버튼과 전자식 차일드 락 버튼 위치를 알려줘.
기대: overview_33 | 의미: overview_34 | 결합: overview_33
질문: 블루투스 핸즈프리 버튼과 음성 인식 버튼은 어디에 있나요?
기대: overview_35 | 의미: overview_34 | 결합: overview_35
질문: 패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.
기대: overview_35 | 의미: overview_34 | 결합: overview_35
질문: 차간거리 버튼과 차로 주행 보조 버튼 위치를 찾아줘.
기대: overview_35 | 의미: overview_34 | 결합: overview_35
질문: 차단 클립을 버클에 꽂으면 경고등과 경고음은 어떻게 되나요?
기대: seatbelt_54 | 의미: overview_34 | 결합: seatbelt_54


### 4. 다른 표현의 질문도 확인

기존 질문만 좋아지면 새로운 표현에서 잘 작동하는지 알기 어렵습니다. 아래 열 질문은 같은 자료를 다른 표현으로 묻는 추가 실험입니다. 이 결과를 보고 검색 비중을 다시 맞추지 않습니다.

In [4]:
# [프로젝트 추가 기록] 수업의 의미 검색과 TF-IDF 개념을 결합한 개인 검색 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
additional_questions = [
    ("전자식 파킹 브레이크 스위치는 안내도에서 어디인가요?", "overview_33"),
    ("문 안쪽 손잡이 위치를 확인하고 싶어요.", "overview_33"),
    ("변속 칼럼과 시동 버튼 위치를 보여주세요.", "overview_34"),
    ("동승석 앞의 글로브 박스는 몇 번으로 표시됐나요?", "overview_34"),
    ("핸들에 달린 음성 인식 버튼을 찾아주세요.", "overview_35"),
    ("오디오를 조절하는 리모컨 스위치는 안내도의 어느 항목인가요?", "overview_35"),
    ("엔진 오일 표에 적힌 0W-20은 어느 엔진 행에 있나요?", "table_44"),
    ("SAE 점도 표의 가장 낮은 온도 눈금은 몇 도인가요?", "table_44"),
    ("안전벨트를 꼰 채 매면 되나요?", "seatbelt_54"),
    ("두 사람이 안전벨트 하나를 같이 써도 되나요?", "seatbelt_54"),
]
assert all(token_count(question) <= token_budget for question, _ in additional_questions)
additional_vectors = embedding_model.encode(
    [question for question, _ in additional_questions],
    convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
)
additional_comparison = compare_questions(additional_questions, additional_vectors)
print_comparison(additional_comparison, "추가 표현 질문")

# 점수가 바뀌어도 원문·그림 연결은 같은 ID로 유지돼야 합니다.
for question, vector in zip([q for q, _ in additional_questions], additional_vectors):
    hits = hybrid_rank(question, vector)
    assert len({h["parent_record"]["metadata"]["record_id"] for h in hits}) == len(hits)
    for hit in hits:
        parent_id = hit["parent_record"]["metadata"]["record_id"]
        assert hit["parent_record"] is parent_lookup[parent_id]

추가 표현 질문 | 질문 수: 10
의미 검색 첫 결과 일치: 8
결합 검색 첫 결과 일치: 10
결합 검색 상위 세 결과 포함: 10
질문: 변속 칼럼과 시동 버튼 위치를 보여주세요.
기대: overview_34 | 의미: overview_33 | 결합: overview_34
질문: 핸들에 달린 음성 인식 버튼을 찾아주세요.
기대: overview_35 | 의미: overview_34 | 결합: overview_35


### 5. 부품 이름 근거와 자료 밖 질문 살펴보기

검색 결과에서 부품 이름이 실제로 일치하는지 확인합니다. 하지만 이름 일치만으로 차량 기능의 상세 조작법까지 알 수 있는 것은 아닙니다. 안내도에는 위치와 참조 쪽수만 있습니다.

자료 밖 질문에도 후보는 생길 수 있습니다. 결합 점수나 이름 일치는 정답 확률이 아니며, 답이 있다는 보증도 아닙니다.

In [5]:
# [프로젝트 추가 기록] 수업의 의미 검색과 TF-IDF 개념을 결합한 개인 검색 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
example_question = "패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘."
hit = hybrid_search(example_question, top_k=1)[0]
parent = hit["parent_record"]
print("질문:", example_question)
print("찾은 원문:", parent["metadata"]["title"])
print("원문과 일치한 부품 이름:", hit["exact_names"])
print("PDF 출처:", parent["metadata"]["source_pages"])
print("이미지 참조:", [(p["pdf_page_number"], p["name"]) for p in parent["image_parts"]])
print("의미 점수:", round(hit["semantic_score"], 4),
      "| 검색어 일치 점수:", round(hit["keyword_score"], 4))

for question in previous["missing_questions"]:
    hit = hybrid_search(question, top_k=1)[0]
    print("\n이번 자료에 답 없는 질문:", question)
    print("반환된 후보:", hit["parent_record"]["metadata"]["title"])
    print("의미:", round(hit["semantic_score"], 4),
          "| 검색어:", round(hit["keyword_score"], 4),
          "| 결합:", round(hit["combined_score"], 4))
print("\n정답·거절 임계값:", "이번 실험에서는 정하지 않음")
print("파일·Supabase 저장:", "실행하지 않음")

질문: 패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.
찾은 원문: 차량 내부 III
원문과 일치한 부품 이름: ['패들 쉬프트']
PDF 출처: [35, 36]
이미지 참조: [(35, 'I1.jpg')]
의미 점수: 0.48 | 검색어 일치 점수: 0.2276

이번 자료에 답 없는 질문: 오늘 서울 날씨와 내일 강수확률을 알려줘.
반환된 후보: 온도에 따른 엔진 오일 SAE 점도 분류표
의미: 0.2921 | 검색어: 0.0 | 결합: 0.146

이번 자료에 답 없는 질문: 차대번호는 차체의 어디에 새겨져 있나요?
반환된 후보: 차량 내부 I
의미: 0.6297 | 검색어: 0.0308 | 결합: 0.3302

이번 자료에 답 없는 질문: 싼타페의 권장 타이어 공기압 수치를 알려줘.
반환된 후보: 온도에 따른 엔진 오일 SAE 점도 분류표
의미: 0.5489 | 검색어: 0.0 | 결합: 0.2744

정답·거절 임계값: 이번 실험에서는 정하지 않음
파일·Supabase 저장: 실행하지 않음


### 이번 실행 결과

기존 20개 질문: 의미 검색의 첫 원문 일치 15개 → 결합 검색 20개.

추가 표현 10개 질문: 의미 검색 8개 → 결합 검색 10개.

기존에 틀린 다섯 질문과 추가 표현의 두 질문이 개선됐습니다. 평가 질문을 바꾸거나 검색 비중을 결과에 맞춰 다시 조정하지 않았습니다. 이 결과는 원문 묶음 5개만 검색한 실험이며, 전체 PDF 또는 생성 답변의 정확도를 뜻하지 않습니다.

자료 밖 질문에도 후보가 반환됩니다. 낮아진 결합 점수만으로 자동 거절 여부를 정하지 않습니다. 다음에는 검색 후보의 원문에 실제 답을 뒷받침하는 정보가 있는지 확인하는 단계가 필요합니다.